# 07 — Photo Readiness Model

This notebook prepares the first visual ML model for the Cane Corso Growth Geometry Lab.

Goal: classify uploaded images as **accepted**, **limited** or **rejected** before any Cane Corso visual match score is shown.


## Why this model exists

A visual comparison can be misleading if the uploaded image is cropped, taken from the wrong angle, too dark, distorted or missing the full dog.

So the first model should answer: **is this photo suitable for comparison?**


In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

labels_path = ROOT / 'data' / 'images' / 'labels' / 'sample-image-labels.csv'
labels_path


In [ ]:
df = pd.read_csv(labels_path)
df.head()


## Create readiness labels

The first prototype can derive a readiness label from existing metadata. Later, this label should be reviewed manually for every real training image.


In [ ]:
def derive_readiness(row):
    ready = str(row.get('comparison_ready', '')).lower()
    quality = str(row.get('photo_quality', '')).lower()
    issues = str(row.get('issues', '')).strip()

    if ready in {'true', 'yes', '1', 'accepted'} and quality in {'good', 'high'} and issues in {'', 'nan', 'none'}:
        return 'accepted'
    if ready in {'false', 'no', '0', 'rejected'}:
        return 'rejected'
    return 'limited'

df['readiness_label'] = df.apply(derive_readiness, axis=1)
df['readiness_label'].value_counts()


## Data readiness check

Before training a real model, each class should have enough examples. For a first baseline, aim for at least 50 images per class. For a stronger experiment, aim for 200+ per class.


In [ ]:
minimum_per_class = 50
counts = df['readiness_label'].value_counts().to_dict()
is_ready = all(counts.get(label, 0) >= minimum_per_class for label in ['accepted', 'limited', 'rejected'])
counts, is_ready


## Baseline training idea

When real images exist, use transfer learning instead of training a CNN from zero. A practical path is:

1. resize images to a fixed size;
2. start with a pretrained MobileNet / EfficientNet / ResNet backbone;
3. train a small classification head for `accepted`, `limited`, `rejected`;
4. evaluate confusion matrix, recall for rejected photos and reliability of limited photos;
5. export only after safety metrics are acceptable.


In [ ]:
report = {
    'model': 'photo_readiness_classifier',
    'status': 'ready_for_baseline_training' if is_ready else 'not_ready_for_training',
    'rows': int(len(df)),
    'class_counts': counts,
    'minimum_per_class': minimum_per_class,
    'safety_policy': {
        'accepted': 'allow comparison with orientation-only disclaimer',
        'limited': 'allow limited comparison only with visible warning',
        'rejected': 'block match score and request a new photo',
    },
}

output_path = ROOT / 'reports' / 'vision' / 'photo-readiness-model-plan.json'
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
report


## Safety note

This readiness model is not a breed-purity model. It only controls whether the photo is suitable for visual comparison.

A rejected photo must not produce a Visual Cane Corso Match score.
